
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [12]:
import os 
from langchain_groq import ChatGroq

os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

model = ChatGroq(
    model="qwen/qwen3.6-27b"
)

In [ ]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")
    cast:list[str]=Field(description="The Cast Of the mocies")

In [14]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.5', 'langchain': '1.3.15'}}, client=<groq.resources.chat.completions.Completions object at 0x000001C1D9722AD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001C1D97234D0>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'integer'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'type': 'number'}, 'cast': {'description': 'The Cast Of the mocies', 'items': {'type': 'string'}, 'type': 'array'}}, 'required': ['title', 'year', 'director', 'rating', 'cast'], 'type': 'object'}}}], 'ls_structured_output_f

In [15]:
model.invoke("Provide details about the movie inception")

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Request**: The user wants details about the movie "Inception". This is a straightforward request for information about a well-known film.\n\n2.  **Identify Key Information Needed**: For a movie, typical details include:\n   - Title\n   - Release year\n   - Director\n   - Writer(s)\n   - Cast\n   - Genre\n   - Plot summary\n   - Themes\n   - Box office/Reception\n   - Awards/Nominations\n   - Notable aspects (cinematography, music, etc.)\n   - Sequels/Related works\n\n3.  **Gather Facts (Internal Knowledge)**:\n   - *Title*: Inception\n   - *Release Year*: 2010\n   - *Director*: Christopher Nolan\n   - *Writer*: Christopher Nolan\n   - *Cast*: Leonardo DiCaprio, Joseph Gordon-Levitt, Elliot Page, Tom Hardy, Ken Watanabe, Dileep Rao, Cillian Murphy, Tom Berenger, Marion Cotillard, Michael Caine\n   - *Genre*: Sci-fi, Action, Thriller, Heist\n   - *Plot Summary*: Cobb (DiCaprio) leads a team of specialists 

In [ ]:
model_with_structure.invoke("Provide details about the movie inception")

Movie(title=2012, year=2009, director='Roland Emmerich', rating=5.8, cast=['John Cusack', 'Amanda Peet', 'Chiwetel Ejiofor', 'Thandiwe Newton', 'Woody Harrelson', 'Oliver Platt', 'Adrien Brody'])

### Mesage output alomgsde parsed Strucure 

In [18]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:** The user is asking for details about the movie "Inception".\n2.  **Identify Required Information:** I need to provide details about the movie. The available tool is `Movie`, which requires:\n   - title (string)\n   - year (integer)\n   - director (string)\n   - rating (number)\n3.  **Retrieve Knowledge about Inception:**\n   - Title: Inception\n   - Year: 2010\n   - Director: Christopher Nolan\n   - Rating: I need a rating out of 10. IMDB rating is 8.8/10. I\'ll use 8.8.\n4.  **Check Tool Requirements:** All required parameters are available.\n   - title: "Inception"\n   - year: 2010\n   - director: "Christopher Nolan"\n   - rating: 8.8\n5.  **Construct Tool Call:** Call the `Movie` function with these parameters.\n6.  **Execute Tool Call:** (Mental simulation of the tool call)\n   `{"name": "Movie", "arguments": {"title": "Inception", "year": 2010, "directo

### Nested Structure 

In [20]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response


MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Michael Caine', role='Miles')], genres=['Action', 'Science Fiction', 'Thriller'], budget=None)

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [21]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [27]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'The Avengers',
 'year': 2012}

In [24]:
model.profile

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [29]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='749834cf-0ad9-4b27-bd1a-edaf1e9f89b7'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Input text: "John Doe, john@example.com, (555) 123-4567"\n   - Task: Extract contact info\n   - Available tool: `ContactInfo` with parameters `name`, `email`, `phone` (all required)\n\n2.  **Identify Required Parameters:**\n   - `name`: "John Doe"\n   - `email`: "john@example.com"\n   - `phone`: "(555) 123-4567"\n\n3.  **Match with Tool:**\n   - The tool `ContactInfo` perfectly matches the required extraction.\n   - All required parameters are present in the input.\n\n4.  **Construct Tool Call:**\n   - Function: `ContactInfo`\n   - Arguments: `{"name": "John Doe", "email": "john@example.com", "phone": "(555) 123-4567"}`\n\n5.  **Execute Tool Call:** (Internal simula

In [30]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [32]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')